<a href="https://colab.research.google.com/github/littlehorsebrother2021/python_code_files/blob/main/Deepseek_Yolov2_Face_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 2. 创建一个统一的原始数据存放目录
!mkdir -p /content/wider_face_raw

# ================= 核心修改区域 =================
# 请分别把下面三个双引号里的路径，替换为你网盘中对应的真实快捷方式路径！

# ① 映射训练集压缩包 (WIDER_train.zip)
!ln -s "/content/drive/MyDrive/WIDER_train.zip" /content/wider_face_raw/WIDER_train.zip

# ② 映射验证集压缩包 (WIDER_val.zip)
!ln -s "/content/drive/MyDrive/WIDER_val.zip" /content/wider_face_raw/WIDER_val.zip

# ③ 映射标注文件解压后的文件夹或压缩包（包含 wider_face_train_bbx_gt.txt 的那个）
!ln -s "/content/drive/MyDrive/wider_face_split.zip" /content/wider_face_raw/wider_face_split.zip
# ===============================================

# 3. 创建本地极速 SSD 目录并解压
!mkdir -p /content/wider_face_local
!unzip -q /content/wider_face_raw/WIDER_train.zip -d /content/wider_face_local/
!unzip -q /content/wider_face_raw/WIDER_val.zip -d /content/wider_face_local/
!unzip -q /content/wider_face_raw/wider_face_split.zip -d /content/wider_face_local/

# 4. 验证文件是否全部就位
print("\n--- 检查本地虚拟盘中的文件布局 ---")
!ls -l /content/wider_face_local/
print("\n--- 检查标注文件是否存在 ---")
!ls -l /content/wider_face_raw/wider_face_split/


--- 检查本地虚拟盘中的文件布局 ---
total 12
drwxr-xr-x 2 root root 4096 Mar 31  2017 wider_face_split
drwxr-xr-x 3 root root 4096 Nov 18  2015 WIDER_train
drwxr-xr-x 3 root root 4096 Nov 18  2015 WIDER_val

--- 检查标注文件是否存在 ---
ls: cannot access '/content/wider_face_raw/wider_face_split/': No such file or directory


In [ ]:
import os
import cv2

def convert_wider_to_yolo(txt_path, img_dir, out_label_dir, out_list_txt, is_test=False):
    """
    Wider Face 标注格式转标准 YOLO 格式脚本
    :param txt_path: 官方标注文本路径（如 wider_face_train_bbx_gt.txt）
    :param img_dir: 对应图片的根目录（如 WIDER_train/images）
    :param out_label_dir: 转换后的 YOLO .txt 标签存放目录
    :param out_list_txt: 生成的路径清单文件（如 face_train.txt），供开源库训练读取
    :param is_test: 是否为测试集（测试集没有公开标注，只需生成路径清单）
    """
    os.makedirs(out_label_dir, exist_ok=True)
    total_images = 0

    with open(out_list_txt, 'w') as list_f:
        # 如果是测试集，没有官方的标注文本，我们直接遍历图片生成路径清单即可
        if is_test or not txt_path or not os.path.exists(txt_path):
            print(f"开始处理测试集图片清单...")
            for root, dirs, files in os.walk(img_dir):
                for file in files:
                    if file.endswith('.jpg') or file.endswith('.png'):
                        # 写入绝对路径
                        list_f.write(os.path.join(root, file) + '\n')
                        total_images += 1
            print(f"测试集处理完成，共计 {total_images} 张图片。")
            return

        print(f"开始解析标注文件: {txt_path}")
        with open(txt_path, 'r') as f:
            lines = f.readlines()

        i = 0
        while i < len(lines):
            line = lines[i].strip()
            # 识别到图片路径行（例如：0--Parade/0_Parade_marchingband_1_849.jpg）
            if line.endswith('.jpg') or line.endswith('.png'):
                img_path_rel = line
                img_path_full = os.path.join(img_dir, img_path_rel)

                # 读取图片以获取分辨率（宽 w 和高 h），用于 YOLO 坐标归一化
                img = cv2.imread(img_path_full)
                if img is None:
                    # 如果图片解压损坏或不存在，跳过对应的数据行
                    i += 1
                    if i < len(lines):
                        num_boxes = int(lines[i].strip())
                        i += num_boxes + 1
                    continue
                h, w, _ = img.shape

                # 将该图片的完整路径写入清单文件
                list_f.write(img_path_full + '\n')
                total_images += 1

                # 生成对应的 YOLO .txt 标签文件路径（保持文件名一致，放进 labels 目录）
                img_name = os.path.basename(img_path_rel)
                label_name = os.path.splitext(img_name)[0] + '.txt'
                label_txt_path = os.path.join(out_label_dir, label_name)

                # 下一行是人脸框的数量
                i += 1
                num_boxes = int(lines[i].strip())

                # 循环读取每一个人脸框坐标
                with open(label_txt_path, 'w') as label_f:
                    for _ in range(num_boxes):
                        i += 1
                        box_line = lines[i].strip().split()
                        if len(box_line) < 4:
                            continue

                        # Wider Face 格式: x1, y1, w, h (绝对像素值)
                        x1, y1, box_w, box_h = list(map(int, box_line[:4]))

                        # 过滤无效的或严重模糊不可见的干扰框
                        if box_w <= 0 or box_h <= 0:
                            continue

                        # 核心转换算法：Wider Face (左上角 x1,y1) -> YOLO (中心点 x_center, y_center) 并归一化到 0~1
                        x_center = (x1 + box_w / 2.0) / w
                        y_center = (y1 + box_h / 2.0) / h
                        norm_w = box_w / w
                        norm_h = box_h / h

                        # 写入标签，因为只有“人脸”一类，所以类别 ID 固定为 0
                        label_f.write(f"0 {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}\n")
            i += 1

    print(f"成功处理完该数据集，转换后共计 {total_images} 张有效图片。")

# ==================== 开始执行转换 ====================
print("--- 1. 开始转换训练集 (Train) ---")
convert_wider_to_yolo(
    txt_path='/content/wider_face_local/wider_face_split/wider_face_train_bbx_gt.txt',
    img_dir='/content/wider_face_local/WIDER_train/images',
    out_label_dir='/content/wider_face_local/WIDER_train/labels',
    out_list_txt='face_train.txt'
)

print("\n--- 2. 开始转换验证集 (Val) ---")
convert_wider_to_yolo(
    txt_path='/content/wider_face_local/wider_face_split/wider_face_val_bbx_gt.txt',
    img_dir='/content/wider_face_local/WIDER_val/images',
    out_label_dir='/content/wider_face_local/WIDER_val/labels',
    out_list_txt='face_val.txt'
)

print("\n--- 3. 开始生成测试集清单 (Test) ---")
# 官方测试集无公开标注，因此 txt_path 传 None，脚本将只批量搜集图片路径用于后续推理评估
convert_wider_to_yolo(
    txt_path=None,
    img_dir='/content/wider_face_local/WIDER_test/images',
    out_label_dir='/content/wider_face_local/WIDER_test/labels',
    out_list_txt='face_test.txt',
    is_test=True
)

print("\n【大功告成】所有格式转换已完成！可以直接进行下一阶段修改 .cfg 文件的操作。")

--- 1. 开始转换训练集 (Train) ---
开始解析标注文件: /content/wider_face_local/wider_face_split/wider_face_train_bbx_gt.txt
成功处理完该数据集，转换后共计 12880 张有效图片。

--- 2. 开始转换验证集 (Val) ---
开始解析标注文件: /content/wider_face_local/wider_face_split/wider_face_val_bbx_gt.txt
成功处理完该数据集，转换后共计 3226 张有效图片。

--- 3. 开始生成测试集清单 (Test) ---
开始处理测试集图片清单...
测试集处理完成，共计 0 张图片。

【大功告成】所有格式转换已完成！可以直接进行下一阶段修改 .cfg 文件的操作。


In [ ]:
!pip install torch torchvision opencv-python numpy

In [ ]:
!ls /content/wider_face_local/WIDER_val/labels/ | head -5

0_Parade_marchingband_1_1004.txt
0_Parade_marchingband_1_1045.txt
0_Parade_marchingband_1_104.txt
0_Parade_marchingband_1_139.txt
0_Parade_marchingband_1_147.txt


In [ ]:
import os
import time
import math
import random
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# ====================== 超参数 ======================
IMG_SIZE = 224
S = 7
B = 5
C = 1
ANCHORS = np.array([
    [30, 40], [50, 70], [80, 100], [120, 160], [200, 240]
], dtype=np.float32) / IMG_SIZE

# ====================== 数据集 ======================
class YoloFaceDataset(Dataset):
    def __init__(self, list_file, label_root, img_size=IMG_SIZE, S=S, B=B, C=C, anchors=ANCHORS):
        with open(list_file, 'r') as f:
            self.lines = [line.strip() for line in f.readlines() if line.strip()]
        self.label_root = label_root
        self.img_size = img_size
        self.S = S
        self.B = B
        self.C = C
        self.anchors = anchors

    def __len__(self):
        return len(self.lines)

    def __getitem__(self, idx):
        img_path = self.lines[idx]
        img_name = os.path.basename(img_path)
        label_name = os.path.splitext(img_name)[0] + '.txt'
        label_path = os.path.join(self.label_root, label_name)

        img = cv2.imread(img_path)
        if img is None:
            return torch.zeros((3, self.img_size, self.img_size)), torch.zeros((self.S, self.S, self.B, 5 + self.C))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.img_size, self.img_size)).astype(np.float32) / 255.0
        img = img.transpose(2, 0, 1)

        target = np.zeros((self.S, self.S, self.B, 5 + self.C), dtype=np.float32)
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                boxes = []
                for line in f.readlines():
                    parts = list(map(float, line.strip().split()))
                    if len(parts) == 5:
                        cls, xc, yc, bw, bh = parts
                        boxes.append([xc, yc, bw, bh])

            for box in boxes:
                xc, yc, bw, bh = box
                col = int(xc * self.S)
                row = int(yc * self.S)
                col = min(self.S - 1, max(0, col))
                row = min(self.S - 1, max(0, row))

                best_iou, best_k = -1, 0
                box_w, box_h = bw * self.img_size, bh * self.img_size
                for k in range(self.B):
                    aw = self.anchors[k, 0] * self.img_size
                    ah = self.anchors[k, 1] * self.img_size
                    inter_w = min(box_w, aw)
                    inter_h = min(box_h, ah)
                    iou = (inter_w * inter_h) / (box_w * box_h + aw * ah - inter_w * inter_h + 1e-6)
                    if iou > best_iou:
                        best_iou, best_k = iou, k

                if target[row, col, best_k, 4] == 0:
                    cell_x = col / self.S
                    cell_y = row / self.S
                    tx = (xc - cell_x) * self.S
                    ty = (yc - cell_y) * self.S
                    tw = math.log(bw / (self.anchors[best_k, 0] + 1e-6))
                    th = math.log(bh / (self.anchors[best_k, 1] + 1e-6))
                    target[row, col, best_k, 0:4] = [tx, ty, tw, th]
                    target[row, col, best_k, 4] = 1.0
                    target[row, col, best_k, 5] = 1.0

        return torch.tensor(img), torch.tensor(target)

# ====================== 模型 ======================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, 1, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.LeakyReLU(0.1, inplace=True)
        )
    def forward(self, x):
        return self.block(x)

class TinyYoloFaceNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(
            ConvBlock(3, 16),   nn.MaxPool2d(2),
            ConvBlock(16, 32),  nn.MaxPool2d(2),
            ConvBlock(32, 64),  nn.MaxPool2d(2),
            ConvBlock(64, 128), nn.MaxPool2d(2),
            ConvBlock(128, 256),nn.MaxPool2d(2),
            ConvBlock(256, 512)
        )
        self.head = nn.Conv2d(512, B * (5 + C), 1)

    def forward(self, x):
        x = self.backbone(x)
        return self.head(x)

# ====================== 损失函数 ======================
class YoloLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.mse = nn.MSELoss(reduction='sum')
        self.bce = nn.BCEWithLogitsLoss(reduction='sum')

    def forward(self, pred, target):
        N = pred.size(0)
        pred = pred.view(N, B, 5+C, S, S).permute(0, 3, 4, 1, 2)
        obj_mask = target[..., 4].bool()
        noobj_mask = ~obj_mask

        coord_loss = 0.0
        if obj_mask.any():
            coord_loss += self.mse(pred[..., 0:2][obj_mask], target[..., 0:2][obj_mask])
            coord_loss += self.mse(pred[..., 2:4][obj_mask], target[..., 2:4][obj_mask])

        conf_loss = 0.0
        if obj_mask.any():
            conf_loss += self.bce(pred[..., 4][obj_mask], torch.ones_like(pred[..., 4][obj_mask]))
        if noobj_mask.any():
            conf_loss += 0.5 * self.bce(pred[..., 4][noobj_mask], torch.zeros_like(pred[..., 4][noobj_mask]))

        cls_loss = 0.0
        if obj_mask.any():
            cls_loss = self.bce(pred[..., 5:][obj_mask], target[..., 5:][obj_mask])

        total = 5.0 * coord_loss + conf_loss + cls_loss
        return total / N

# ====================== 训练 ======================
def train():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")

    train_dataset = YoloFaceDataset(
        list_file='/content/face_train.txt',
        label_root='/content/wider_face_local/WIDER_train/labels'
    )
    val_dataset = YoloFaceDataset(
        list_file='/content/face_val.txt',
        label_root='/content/wider_face_local/WIDER_val/labels'
    )

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, drop_last=False)

    model = TinyYoloFaceNet().to(device)
    criterion = YoloLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

    # 学习率调度器（移除 verbose 参数）
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3
    )

    # 早停参数
    MAX_EPOCHS = 50
    EARLY_STOP_PATIENCE = 8
    best_val_loss = float('inf')
    epochs_no_improve = 0
    current_lr = optimizer.param_groups[0]['lr']  # 记录初始学习率

    print("Start training...")
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        train_loss = 0.0
        for imgs, targets in train_loader:
            imgs, targets = imgs.to(device), targets.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for imgs, targets in val_loader:
                imgs, targets = imgs.to(device), targets.to(device)
                val_loss += criterion(model(imgs), targets).item()

        avg_train = train_loss / len(train_loader)
        avg_val = val_loss / len(val_loader)
        print(f"Epoch [{epoch}/{MAX_EPOCHS}] Train Loss: {avg_train:.4f}, Val Loss: {avg_val:.4f}")

        # 学习率衰减
        scheduler.step(avg_val)

        # 手动检查学习率变化并打印
        new_lr = optimizer.param_groups[0]['lr']
        if new_lr != current_lr:
            print(f"  -> learning rate reduced to {new_lr:.2e}")
            current_lr = new_lr

        # 早停逻辑
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(model.state_dict(), 'best_tiny_yolo_face.pth')
            print(f"  -> saved best model (val_loss: {best_val_loss:.4f})")
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            print(f"  -> no improvement for {epochs_no_improve} epoch(s)")

        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(f"\nEarly stopping triggered at epoch {epoch}. Best val loss: {best_val_loss:.4f}")
            break

    # ====================== 导出与校准 ======================
    model.load_state_dict(torch.load('best_tiny_yolo_face.pth'))
    model.to('cpu').eval()

    # 融合 BN（可选）
    try:
        model = torch.quantization.fuse_modules(model, [
            ['backbone.0.block.0', 'backbone.0.block.1', 'backbone.0.block.2'],
            ['backbone.2.block.0', 'backbone.2.block.1', 'backbone.2.block.2'],
            ['backbone.4.block.0', 'backbone.4.block.1', 'backbone.4.block.2'],
            ['backbone.6.block.0', 'backbone.6.block.1', 'backbone.6.block.2'],
            ['backbone.8.block.0', 'backbone.8.block.1', 'backbone.8.block.2'],
            ['backbone.10.block.0', 'backbone.10.block.1', 'backbone.10.block.2']
        ])
        print("BN fused")
    except Exception as e:
        print("BN fusion skipped:", e)

    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
    torch.onnx.export(model, dummy, 'tiny_yolo_face.onnx',
                      opset_version=18,
                      input_names=['input'], output_names=['output'],
                      do_constant_folding=True)
    print("ONNX exported.")

    print("Generating calibration data (NHWC format)...")
    with open('/content/face_train.txt', 'r') as f:
        all_paths = [l.strip() for l in f if l.strip()]
    random.shuffle(all_paths)
    calib_imgs = []
    for p in all_paths[:200]:
        img = cv2.imread(p)
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE)).astype(np.float32) / 255.0
        calib_imgs.append(img)
    calib_array = np.stack(calib_imgs, axis=0)
    np.savez('/content/calib_data.npz', calibration=calib_array)
    print(f"Calibration data saved: {calib_array.shape}")

if __name__ == '__main__':
    train()

Device: cuda
Start training...
Epoch [1/50] Train Loss: 39.7085, Val Loss: 28.7604
  -> saved best model (val_loss: 28.7604)
Epoch [2/50] Train Loss: 25.8156, Val Loss: 28.8886
  -> no improvement for 1 epoch(s)
Epoch [3/50] Train Loss: 22.2973, Val Loss: 24.3034
  -> saved best model (val_loss: 24.3034)
Epoch [4/50] Train Loss: 20.3797, Val Loss: 23.0473
  -> saved best model (val_loss: 23.0473)
Epoch [5/50] Train Loss: 18.8522, Val Loss: 21.5370
  -> saved best model (val_loss: 21.5370)
Epoch [6/50] Train Loss: 17.4641, Val Loss: 20.3284
  -> saved best model (val_loss: 20.3284)
Epoch [7/50] Train Loss: 16.3344, Val Loss: 24.5207
  -> no improvement for 1 epoch(s)
Epoch [8/50] Train Loss: 15.0834, Val Loss: 17.9619
  -> saved best model (val_loss: 17.9619)
Epoch [9/50] Train Loss: 14.7198, Val Loss: 18.3724
  -> no improvement for 1 epoch(s)
Epoch [10/50] Train Loss: 13.4855, Val Loss: 19.2497
  -> no improvement for 2 epoch(s)
Epoch [11/50] Train Loss: 12.7434, Val Loss: 16.6888
  -

ModuleNotFoundError: No module named 'onnxscript'

Device: cuda  
Start training...  
Epoch [1/50] Train Loss: 39.7085, Val Loss: 28.7604  
  -> saved best model (val_loss: 28.7604)  
Epoch [2/50] Train Loss: 25.8156, Val Loss: 28.8886  
  -> no improvement for 1 epoch(s)  
Epoch [3/50] Train Loss: 22.2973, Val Loss: 24.3034  
  -> saved best model (val_loss: 24.3034)  
Epoch [4/50] Train Loss: 20.3797, Val Loss: 23.0473  
  -> saved best model (val_loss: 23.0473)  
Epoch [5/50] Train Loss: 18.8522, Val Loss: 21.5370  
  -> saved best model (val_loss: 21.5370)  
Epoch [6/50] Train Loss: 17.4641, Val Loss: 20.3284  
  -> saved best model (val_loss: 20.3284)  
Epoch [7/50] Train Loss: 16.3344, Val Loss: 24.5207  
  -> no improvement for 1 epoch(s)  
Epoch [8/50] Train Loss: 15.0834, Val Loss: 17.9619  
  -> saved best model (val_loss: 17.9619)  
Epoch [9/50] Train Loss: 14.7198, Val Loss: 18.3724  
  -> no improvement for 1 epoch(s)  
Epoch [10/50] Train Loss: 13.4855, Val Loss: 19.2497  
  -> no improvement for 2 epoch(s)  
Epoch [11/50] Train Loss: 12.7434, Val Loss: 16.6888  
  -> saved best model (val_loss: 16.6888)  
Epoch [12/50] Train Loss: 11.5883, Val Loss: 20.4058
  -> no improvement for 1 epoch(s)  
Epoch [13/50] Train Loss: 10.9379, Val Loss: 17.9525  
  -> no improvement for 2 epoch(s)  
Epoch [14/50] Train Loss: 10.0705, Val Loss: 18.2749  
  -> no improvement for 3 epoch(s)  
Epoch [15/50] Train Loss: 9.6918, Val Loss: 18.4236  
  -> learning rate reduced to 5.00e-04  
  -> no improvement for 4 epoch(s)  
Epoch [16/50] Train Loss: 7.6033, Val Loss: 19.0459  
  -> no improvement for 5 epoch(s)  
Epoch [17/50] Train Loss: 6.8010, Val Loss: 18.0119  
  -> no improvement for 6 epoch(s)  
Epoch [18/50] Train Loss: 6.3430, Val Loss: 20.4702  
  -> no improvement for 7 epoch(s)  
Epoch [19/50] Train Loss: 5.8921, Val Loss: 19.6358  
  -> learning rate reduced to 2.50e-04  
  -> no improvement for 8 epoch(s)  

下面针对停滞验证损失进行进一步训练

In [ ]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader

# 假设之前的数据集类、模型类、损失函数类已经定义（已在同一notebook中运行过）
# 如未定义，请重新运行包含这些类定义的单元格

# ---------- 配置继续训练的参数 ----------
CONTINUE_EPOCHS = 30          # 继续训练的轮数（设大一点，靠早停提前终止）
LR = 1e-4                     # 更低的初始学习率
SCHEDULER_PATIENCE = 5        # 学习率衰减耐心值（比之前稍大）
EARLY_STOP_PATIENCE = 10      # 早停耐心值（适当放宽，给模型更多机会）

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# 加载之前保存的最佳模型权重（epoch 11）
model = TinyYoloFaceNet().to(device)
model.load_state_dict(torch.load('best_tiny_yolo_face.pth', map_location=device))

criterion = YoloLoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=SCHEDULER_PATIENCE
)

# 数据加载器（沿用之前的）
train_dataset = YoloFaceDataset(
    list_file='/content/face_train.txt',
    label_root='/content/wider_face_local/WIDER_train/labels'
)
val_dataset = YoloFaceDataset(
    list_file='/content/face_val.txt',
    label_root='/content/wider_face_local/WIDER_val/labels'
)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, drop_last=False)

# 早停状态初始化
best_val_loss = 16.6888       # 当前最佳（可从之前日志确认）
epochs_no_improve = 0
current_lr = LR

print(f"Continue training from best model (val_loss={best_val_loss:.4f}), LR={LR:.1e}")
for epoch in range(1, CONTINUE_EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for imgs, targets in train_loader:
        imgs, targets = imgs.to(device), targets.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), targets)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for imgs, targets in val_loader:
            imgs, targets = imgs.to(device), targets.to(device)
            val_loss += criterion(model(imgs), targets).item()

    avg_train = train_loss / len(train_loader)
    avg_val = val_loss / len(val_loader)
    print(f"Epoch [{epoch}/{CONTINUE_EPOCHS}] Train Loss: {avg_train:.4f}, Val Loss: {avg_val:.4f}")

    scheduler.step(avg_val)
    new_lr = optimizer.param_groups[0]['lr']
    if new_lr < current_lr:
        print(f"  -> learning rate reduced to {new_lr:.2e}")
        current_lr = new_lr

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), 'best_tiny_yolo_face_continued.pth')  # 新文件名，避免覆盖原最佳
        print(f"  -> saved new best model (val_loss: {best_val_loss:.4f})")
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        print(f"  -> no improvement for {epochs_no_improve} epoch(s)")

    if epochs_no_improve >= EARLY_STOP_PATIENCE:
        print(f"\nEarly stopping triggered at epoch {epoch}. Best val loss: {best_val_loss:.4f}")
        break

print("\n--- Training completed ---")

# 可选：导出 ONNX 和校准数据（如果需要重新生成）
export_assets = False  # 改为 True 以重新导出
if export_assets:
    model.load_state_dict(torch.load('best_tiny_yolo_face_continued.pth'))
    model.to('cpu').eval()
    dummy = torch.randn(1, 3, 224, 224)
    torch.onnx.export(model, dummy, 'tiny_yolo_face_continued.onnx',
                      opset_version=18, input_names=['input'], output_names=['output'],
                      do_constant_folding=True)
    print("ONNX exported.")
    # 校准数据生成（略）

Device: cuda
Continue training from best model (val_loss=16.6888), LR=1.0e-04
Epoch [1/30] Train Loss: 9.6465, Val Loss: 15.6721
  -> saved new best model (val_loss: 15.6721)
Epoch [2/30] Train Loss: 8.9459, Val Loss: 15.9136
  -> no improvement for 1 epoch(s)
Epoch [3/30] Train Loss: 8.5278, Val Loss: 16.0416
  -> no improvement for 2 epoch(s)
Epoch [4/30] Train Loss: 8.1921, Val Loss: 16.1947
  -> no improvement for 3 epoch(s)
Epoch [5/30] Train Loss: 7.8205, Val Loss: 16.4510
  -> no improvement for 4 epoch(s)
Epoch [6/30] Train Loss: 7.5247, Val Loss: 16.3720
  -> no improvement for 5 epoch(s)
Epoch [7/30] Train Loss: 7.2483, Val Loss: 16.5372
  -> learning rate reduced to 5.00e-05
  -> no improvement for 6 epoch(s)
Epoch [8/30] Train Loss: 6.7956, Val Loss: 16.6665
  -> no improvement for 7 epoch(s)
Epoch [9/30] Train Loss: 6.5942, Val Loss: 16.8294
  -> no improvement for 8 epoch(s)
Epoch [10/30] Train Loss: 6.4453, Val Loss: 16.8530
  -> no improvement for 9 epoch(s)
Epoch [11/3

Device: cuda  
Continue training from best model (val_loss=16.6888), LR=1.0e-04    
Epoch [1/30] Train Loss: 9.6465, Val Loss: 15.6721  
  -> saved new best model (val_loss: 15.6721)  
Epoch [2/30] Train Loss: 8.9459, Val Loss: 15.9136  
  -> no improvement for 1 epoch(s)  
Epoch [3/30] Train Loss: 8.5278, Val Loss: 16.0416  
  -> no improvement for 2 epoch(s)  
Epoch [4/30] Train Loss: 8.1921, Val Loss: 16.1947  
  -> no improvement for 3 epoch(s)  
Epoch [5/30] Train Loss: 7.8205, Val Loss: 16.4510  
  -> no improvement for 4 epoch(s)  
Epoch [6/30] Train Loss: 7.5247, Val Loss: 16.3720  
  -> no improvement for 5 epoch(s)  
Epoch [7/30] Train Loss: 7.2483, Val Loss: 16.5372  
  -> learning rate reduced to 5.00e-05  
  -> no improvement for 6 epoch(s)  
Epoch [8/30] Train Loss: 6.7956, Val Loss: 16.6665  
  -> no improvement for 7 epoch(s)  
Epoch [9/30] Train Loss: 6.5942, Val Loss: 16.8294  
  -> no improvement for 8 epoch(s)  
Epoch [10/30] Train Loss: 6.4453, Val Loss: 16.8530  
  -> no improvement for 9 epoch(s)  
Epoch [11/30] Train Loss: 6.2970, Val Loss: 16.9133  
  -> no improvement for 10 epoch(s)  
  
Early stopping triggered at epoch 11. Best val loss: 15.6721  

--- Training completed ---  

下面对模型进行导出

In [ ]:
!pip install onnx onnxscript -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 18.4 MB/s eta 0:00:00


In [ ]:
import torch
import numpy as np
import cv2
import random
import os

# ---------- 模型定义（与训练时完全一致） ----------
class ConvBlock(torch.nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = torch.nn.Sequential(
            torch.nn.Conv2d(in_ch, out_ch, 3, 1, 1, bias=False),
            torch.nn.BatchNorm2d(out_ch),
            torch.nn.LeakyReLU(0.1, inplace=True)
        )
    def forward(self, x):
        return self.block(x)

class TinyYoloFaceNet(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = torch.nn.Sequential(
            ConvBlock(3, 16),   torch.nn.MaxPool2d(2),
            ConvBlock(16, 32),  torch.nn.MaxPool2d(2),
            ConvBlock(32, 64),  torch.nn.MaxPool2d(2),
            ConvBlock(64, 128), torch.nn.MaxPool2d(2),
            ConvBlock(128, 256),torch.nn.MaxPool2d(2),
            ConvBlock(256, 512)
        )
        self.head = torch.nn.Conv2d(512, 5 * (5 + 1), 1)  # B=5, C=1

    def forward(self, x):
        x = self.backbone(x)
        return self.head(x)

# ---------- 加载最佳模型（方案B保存的新最佳） ----------
device = torch.device('cpu')
model = TinyYoloFaceNet().to(device)
# 如果你使用的是方案B保存的模型，文件名应为 'best_tiny_yolo_face_continued.pth'
# 如果方案B没有保存新模型，则回退到原来的 'best_tiny_yolo_face.pth'
model_path = 'best_tiny_yolo_face_continued.pth' if os.path.exists('best_tiny_yolo_face_continued.pth') else 'best_tiny_yolo_face.pth'
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()
print(f"Loaded model from {model_path}")

# ---------- 导出 ONNX ----------
dummy_input = torch.randn(1, 3, 224, 224, device=device)
torch.onnx.export(
    model, dummy_input, 'tiny_yolo_face.onnx',
    opset_version=18,
    input_names=['input'],
    output_names=['output'],
    do_constant_folding=True
)
print("✅ ONNX 导出成功：tiny_yolo_face.onnx")

# ---------- 生成校准数据（NHWC，.npz）----------
print("Generating calibration data (NHWC)...")
with open('/content/face_train.txt', 'r') as f:
    all_paths = [line.strip() for line in f.readlines() if line.strip()]
random.shuffle(all_paths)
calib_imgs = []
for p in all_paths[:200]:
    img = cv2.imread(p)
    if img is None:
        continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224)).astype(np.float32) / 255.0  # HWC
    calib_imgs.append(img)
calib_array = np.stack(calib_imgs, axis=0)
np.savez('/content/calib_data.npz', calibration=calib_array)
print(f"✅ 校准数据已保存：/content/calib_data.npz，形状：{calib_array.shape}")

Loaded model from best_tiny_yolo_face_continued.pth
[torch.onnx] Obtain model graph for `TinyYoloFaceNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `TinyYoloFaceNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
✅ ONNX 导出成功：tiny_yolo_face.onnx
Generating calibration data (NHWC)...
✅ 校准数据已保存：/content/calib_data.npz，形状：(200, 224, 224, 3)


In [ ]:
import onnx
from onnx.external_data_helper import convert_model_to_external_data, convert_model_from_external_data
import os

# 加载拆分模型（包含外部数据）
model = onnx.load('tiny_yolo_face.onnx')

# 检查是否有外部数据
if model.ByteSize() < os.path.getsize('tiny_yolo_face.onnx'):
    print("模型已包含内嵌权重，无需合并。")
else:
    # 将外部数据合并回模型，并保存为新的单文件
    convert_model_from_external_data(model)
    onnx.save(model, 'tiny_yolo_face_single.onnx')
    print(f"合并完成：tiny_yolo_face_single.onnx（大小：{os.path.getsize('tiny_yolo_face_single.onnx') / 1024 / 1024:.2f} MB）")

合并完成：tiny_yolo_face_single.onnx（大小：6.09 MB）


校准数据有问题重新生成

In [ ]:
import numpy as np
import cv2
import random

print("生成 NCHW 校准数据...")
with open('/content/face_train.txt', 'r') as f:
    all_paths = [line.strip() for line in f.readlines() if line.strip()]
random.shuffle(all_paths)

calib_imgs = []
for p in all_paths[:200]:
    img = cv2.imread(p)
    if img is None:
        continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224)).astype(np.float32) / 255.0
    # 关键：从 HWC 转为 CHW
    img = img.transpose(2, 0, 1)  # (3, 224, 224)
    calib_imgs.append(img)

calib_array = np.stack(calib_imgs, axis=0)  # (200, 3, 224, 224)
np.savez('/content/calib_data.npz', calibration=calib_array)
print(f"校准数据已更新：形状 {calib_array.shape}（NCHW）")

生成 NCHW 校准数据...
校准数据已更新：形状 (200, 3, 224, 224)（NCHW）


In [ ]:
import torch
import torch.nn as nn
import cv2
import numpy as np
import matplotlib.pyplot as plt
import random
from torchvision.ops import nms

# ================== 1. 重新定义模型（确保与训练时完全一致） ==================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, 1, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.LeakyReLU(0.1, inplace=True)
        )
    def forward(self, x):
        return self.block(x)

class TinyYoloFaceNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(
            ConvBlock(3, 16),   nn.MaxPool2d(2),
            ConvBlock(16, 32),  nn.MaxPool2d(2),
            ConvBlock(32, 64),  nn.MaxPool2d(2),
            ConvBlock(64, 128), nn.MaxPool2d(2),
            ConvBlock(128, 256),nn.MaxPool2d(2),
            ConvBlock(256, 512)
        )
        self.head = nn.Conv2d(512, 5 * (5 + 1), 1)   # B=5, C=1

    def forward(self, x):
        x = self.backbone(x)
        return self.head(x)

# ================== 2. 解码函数（将网络输出转换为框） ==================
def decode_predictions(output, anchors, S=7, num_classes=1, conf_thresh=0.03, img_size=224):
    device = output.device
    batch_size = output.size(0)  # 应为 1
    output = output.view(batch_size, 5, 5+num_classes, S, S).permute(0,3,4,1,2).contiguous()

    grid_y, grid_x = torch.meshgrid(torch.arange(S, device=device, dtype=torch.float32),
                                    torch.arange(S, device=device, dtype=torch.float32))
    grid = torch.stack((grid_x, grid_y), dim=-1).unsqueeze(2).unsqueeze(0)  # [1,7,7,1,2]

    tx = torch.sigmoid(output[..., 0])
    ty = torch.sigmoid(output[..., 1])
    tw = output[..., 2]
    th = output[..., 3]
    conf = torch.sigmoid(output[..., 4])
    cls_prob = torch.sigmoid(output[..., 5])  # 仅一类

    anchors_t = torch.from_numpy(anchors).float().to(device).view(1,1,1,5,2)
    cx = (grid[..., 0] + tx) / S
    cy = (grid[..., 1] + ty) / S
    w = torch.exp(tw) * anchors_t[..., 0]
    h = torch.exp(th) * anchors_t[..., 1]

    x1 = cx - w/2
    y1 = cy - h/2
    x2 = cx + w/2
    y2 = cy + h/2

    boxes = torch.stack([x1, y1, x2, y2], dim=-1)  # [1,7,7,5,4]
    scores = conf * cls_prob                       # [1,7,7,5]

    # 展平、过滤低分框
    boxes = boxes.reshape(-1, 4) * img_size
    scores = scores.reshape(-1)
    keep = scores > conf_thresh
    boxes, scores = boxes[keep], scores[keep]

    if boxes.numel() == 0:
        return torch.empty(0,4), torch.empty(0)

    # NMS
    keep_idx = nms(boxes, scores, iou_threshold=0.5)
    return boxes[keep_idx], scores[keep_idx]

# ================== 3. 加载模型与anchor ==================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TinyYoloFaceNet().to(device)
model.load_state_dict(torch.load('best_tiny_yolo_face.pth', map_location=device))
model.eval()

ANCHORS = np.array([[30,40],[50,70],[80,100],[120,160],[200,240]], dtype=np.float32) / 224.0

# ================== 4. 随机挑选验证集图片并可视化 ==================
with open('/content/face_val.txt', 'r') as f:
    val_lines = [l.strip() for l in f if l.strip()]
random.shuffle(val_lines)
test_paths = val_lines[:5]

fig, axes = plt.subplots(1, 5, figsize=(20, 8))
for ax, img_path in zip(axes, test_paths):
    img = cv2.imread(img_path)
    if img is None:
        ax.axis('off')
        continue
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    orig_h, orig_w = img_rgb.shape[:2]

    # 预处理（同训练）
    img_resized = cv2.resize(img_rgb, (224, 224)).astype(np.float32) / 255.0
    img_tensor = torch.from_numpy(img_resized).permute(2,0,1).unsqueeze(0).to(device)

    with torch.no_grad():
        pred = model(img_tensor)
        boxes, scores = decode_predictions(pred, ANCHORS, conf_thresh=0.3)

    # 缩放框回原图并绘制
    for box in boxes.cpu().numpy():
        x1, y1, x2, y2 = box
        # 映射到原始尺寸
        x1 = x1 * orig_w / 224
        y1 = y1 * orig_h / 224
        x2 = x2 * orig_w / 224
        y2 = y2 * orig_h / 224
        cv2.rectangle(img_rgb, (int(x1), int(y1)), (int(x2), int(y2)), (0,255,0), 2)

    ax.imshow(img_rgb)
    ax.set_title(os.path.basename(img_path)[:15])
    ax.axis('off')
plt.tight_layout()
plt.show()